## PULIZIA DEL DATASET

### Importazione librerie e caricamento dataset
Iniziamo la fase di Data Cleaning importando le librerie necessarie e caricando il dataset originale. L'obiettivo di questo notebook è applicare le trasformazioni decise durante l'analisi esplorativa (EDA) per preparare i dati all'addestramento del modello predittivo

In [1]:
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
import os

# Mostra tutte le colonne quando stampiamo il dataframe
pd.set_option('display.max_columns', None)

file_path = 'datasets/raw_dataset.csv'

if os.path.exists(file_path):
    df = pd.read_csv(file_path)
    print(f"Caricamento del dataset avvenuto con successo! Dimensioni: {df.shape}")
else:
    print("ERRORE: Il dataset non è stato trovato.")

Caricamento del dataset avvenuto con successo! Dimensioni: (4424, 35)


### Gestione del Target: Pulizia e Codifica Binaria
Come stabilito nelle conclusioni dell'analisi, la classe 'Enrolled' rappresenta uno stato transitorio che vogliamo escludere. Procediamo a filtrare il dataset mantenendo solo i casi certi e convertiamo il Target in una variabile numerica binaria (0 per Dropout, 1 per Graduate). In questo modo abbiamo eliminato quasi 800 studenti

In [2]:
# Filtriamo il dataset per escludere gli 'Enrolled'
df = df[df['Target'] != 'Enrolled'].copy()

# Mappiamo in binario: Dropout = 0, Graduate = 1
df['Target'] = df['Target'].map({'Dropout': 0, 'Graduate': 1})

print(f"Dimensioni del dataset dopo la rimozione degli Enrolled: {df.shape}")
print("\nDistribuzione del nuovo Target binario:")
print(df['Target'].value_counts())

Dimensioni del dataset dopo la rimozione degli Enrolled: (3630, 35)

Distribuzione del nuovo Target binario:
Target
1    2209
0    1421
Name: count, dtype: int64


### Feature Selection: Rimozione colonne ridondanti e rumore
Ora procediamo a snellire il dataset eliminando le variabili che l'analisi ha rivelato essere non ottimali. Rimuoviamo le feature a bassissima varianza (i crediti riconosciuti e le unità senza valutazione), le ridondanze strutturali (Nacionality) e le variabili anagrafiche/macroeconomiche con correlazione pressoché nulla (Marital status, Application order, Course, GDP, Inflation rate, Unemployment rate)

In [4]:
# Lista esatta delle colonne da eliminare
colonne_da_scartare = [
    'Marital status', 
    'Application order', 
    'Course', 
    'Nacionality',
    'Curricular units 1st sem (credited)', 
    'Curricular units 1st sem (without evaluations)',
    'Curricular units 2nd sem (credited)', 
    'Curricular units 2nd sem (without evaluations)',
    'Unemployment rate', 
    'Inflation rate', 
    'GDP'
]

# Droppiamo le colonne
df = df.drop(columns=colonne_da_scartare)

print(f"Dimensioni del dataset dopo la pulizia delle feature: {df.shape}")

Dimensioni del dataset dopo la pulizia delle feature: (3630, 24)


### Feature Engineering: Binning del Background Familiare
Le variabili relative all'occupazione e alla qualifica dei genitori presentano un'elevata cardinalità (decine di categorie diverse descritte nella legenda UCI). Per evitare di disperdere il potere predittivo del modello, procediamo ad aggregare questi codici in tre macro-categorie per l'istruzione (Higher Education, Secondary Education, Basic/Unknown) e tre per l'occupazione (Highly Skilled, Skilled/Clerical, Unskilled/Inactive)

In [ ]:
def group_qualifications(code):
    # Codici per Lauree, Master, Dottorati, ecc.
    higher_edu = [2, 3, 4, 5, 6, 40, 41, 42, 43, 44]
    # Codici per Scuole superiori e corsi professionalizzanti
    secondary_edu = [1, 9, 10, 11, 12, 13, 14, 18, 19, 20, 22, 25, 26, 27, 29, 30, 31, 33]
    
    if code in higher_edu:
        return 'Higher_Education'
    elif code in secondary_edu:
        return 'Secondary_Education'
    else:
        return 'Basic_or_Unknown'

df["Mother's qualification"] = df["Mother's qualification"].apply(group_qualifications)
df["Father's qualification"] = df["Father's qualification"].apply(group_qualifications)


# --- RAGGRUPPAMENTO OCCUPAZIONI (Lavoro) ---
def group_occupations(code):
    #Dirigenti, medici, ingegneri, professori, tecnici specializzati
    highly_skilled = [1, 2, 3, 112, 114, 121, 122, 123, 124, 125, 131, 132, 134, 135]
    #Impiegati, venditori, artigiani, operai specializzati, forze armate
    skilled = [4, 5, 6, 7, 8, 10, 101, 102, 103, 141, 143, 144, 151, 152, 153, 154, 161, 163, 171, 172, 174, 175, 181, 182, 183]
    
    if code in highly_skilled:
        return 'Highly_Skilled'
    elif code in skilled:
        return 'Skilled_or_Clerical'
    else:
        #Lavori non qualificati, studenti, pensionati, disoccupati (0, 9, 90, 99, 191-195)
        return 'Unskilled_or_Inactive'

df["Mother's occupation"] = df["Mother's occupation"].apply(group_occupations)
df["Father's occupation"] = df["Father's occupation"].apply(group_occupations)

print("Raggruppamento delle professioni e qualifiche completato!\n")

print("--- Distribuzione Qualifiche (Titoli di studio) ---")
print("Madre:")
print(df["Mother's qualification"].value_counts())
print("\nPadre:")
print(df["Father's qualification"].value_counts())

print("\n---------------------------------------------------")

print("\nDistribuzione Occupazioni (Lavoro)")
print("Madre:")
print(df["Mother's occupation"].value_counts())
print("\nPadre:")
print(df["Father's occupation"].value_counts())

Raggruppamento delle professioni e qualifiche completato!

--- Distribuzione Qualifiche (Titoli di studio) ---
Madre:
Mother's qualification
Secondary_Education    2680
Basic_or_Unknown        499
Higher_Education        451
Name: count, dtype: int64

Padre:
Father's qualification
Secondary_Education    2615
Basic_or_Unknown        703
Higher_Education        312
Name: count, dtype: int64

---------------------------------------------------

Distribuzione Occupazioni (Lavoro)
Madre:
Mother's occupation
Skilled_or_Clerical      2992
Highly_Skilled            470
Unskilled_or_Inactive     168
Name: count, dtype: int64

Padre:
Father's occupation
Skilled_or_Clerical      2638
Unskilled_or_Inactive     609
Highly_Skilled            383
Name: count, dtype: int64


### Esportazione del Dataset Pulito
Tutte le operazioni di Data Cleaning e Feature Engineering sono concluse. Il dataset è ora privo di ridondanze, rumore e variabili ad alta cardinalità. Procediamo a salvare questa versione definitiva in un nuovo file CSV, che fungerà da base per la successiva fase di addestramento del modello (Model Training)

In [7]:
# Sostituisci con il percorso corretto se usi una cartella specifica, es. 'datasets/dataset_cleaned.csv'
percorso_salvataggio = 'datasets/cleaned_dataset.csv'

# Salviamo il dataframe ignorando l'indice di Pandas (importantissimo per non creare colonne extra)
df.to_csv(percorso_salvataggio, index=False)

print(f"Operazione conclusa con successo!")
print(f"Il dataset pulito ({df.shape[0]} righe, {df.shape[1]} colonne) è stato salvato in: {percorso_salvataggio}")

Operazione conclusa con successo!
Il dataset pulito (3630 righe, 24 colonne) è stato salvato in: datasets/cleaned_dataset.csv
